In [1]:
from google.colab import userdata, drive
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoTokenizer
from itertools import chain


In [3]:
HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)

In [4]:
drive.mount('drive')

Mounted at drive


In [6]:
dataset = load_dataset("azizdevlab/uzbek_corpus")
print(dataset)

README.md:   0%|          | 0.00/321 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/220M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/92.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3011581 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 3011581
    })
})


In [7]:
tokenizer = AutoTokenizer.from_pretrained("azizdevlab/gpt2-small-uzbek")

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
def tokenize_function(example):
    return tokenizer(text=example["text"])
tokenized_ds = dataset.map(tokenize_function,batched=True,remove_columns='text')

Map:   0%|          | 0/3011581 [00:00<?, ? examples/s]

In [10]:
save_path = '/content/drive/MyDrive/uzbek_corpus/tokenized_ds'
tokenized_ds.save_to_disk(save_path)

Saving the dataset (0/6 shards):   0%|          | 0/3011581 [00:00<?, ? examples/s]

In [13]:
def concat(examples):
    examples["input_ids"]=[list(chain.from_iterable(examples['input_ids']))] # convert chain to list of tokens
    examples["attention_mask"]=[list(chain.from_iterable(examples['attention_mask']))] # convert chain to list of tokens
    return examples

concated_ds = tokenized_ds.map(concat,batched=True,num_proc=4)

Map (num_proc=4):   0%|          | 0/3011581 [00:00<?, ? examples/s]

In [14]:
concated_ds.save_to_disk('/content/drive/MyDrive/uzbek_corpus/concated_ds')

Saving the dataset (0/6 shards):   0%|          | 0/3012 [00:00<?, ? examples/s]

In [15]:
def chunk(examples):
    chunk_size = 1024 # modify this accordingly
    input_ids = examples["input_ids"][0] # List[List], pass the inner list
    attention_mask = examples["attention_mask"][0] # List[List]
    input_ids_truncated = []
    attention_mask_truncated = []

    #slice with step_size=chunk_size
    for i in range(0,len(input_ids),chunk_size):
        chunk = input_ids[i:i+chunk_size]
        if len(chunk)==chunk_size: # drop the last chunk if not equal
            input_ids_truncated.append(chunk)
            attention_mask_truncated.append(attention_mask[i:i+chunk_size])
    examples['input_ids']=input_ids_truncated
    examples["attention_mask"]=attention_mask_truncated

    return examples

In [16]:
chunked_ds = concated_ds.map(chunk,batched=True,batch_size=2,num_proc=2)

Map (num_proc=2):   0%|          | 0/3012 [00:00<?, ? examples/s]

In [17]:
chunked_ds.save_to_disk('/content/drive/MyDrive/uzbek_corpus/chunked_ds')

Saving the dataset (0/3 shards):   0%|          | 0/258800 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer,mlm=False)